# Thai DSR — joint mapper + HiFi-GAN บน CUDA

เปิดแท็บเบราว์เซอร์ไว้และอย่าให้แล็ปท็อป sleep ระหว่างฝึก: Free/Pro Colab มีข้อจำกัดเรื่องเวลารัน/idle และอาจยุติ runtime ได้ ([Colab FAQ](https://research.google.com/colaboratory/faq.html)).



อัปโหลด `colab_transfer_bundle.zip` ไปที่ **My Drive/thai_dsr_colab/** เลือก **Runtime → Change runtime type → GPU** แล้วรัน setup ตามลำดับ. Smoke ใหม่ใช้ clip=1, warmup=100, ramp=200, content=0.5 และ 500 steps. Full run ใช้ clip=1, warmup=1000, ramp=1000, content=0.5 (2,000 steps จึงครบ ramp).

In [ ]:
# 1. GPU
import torch
print('torch:', torch.__version__)
print('torch.cuda.is_available():', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select a GPU runtime, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', torch.cuda.get_device_properties(0).total_memory / 2**30)


In [ ]:
# 2. Google Drive
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/thai_dsr_colab')  # EDIT if needed
BUNDLE = DRIVE_ROOT / 'colab_transfer_bundle.zip'
CHECKPOINT_ROOT = DRIVE_ROOT / 'checkpoints'
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
assert BUNDLE.is_file(), f'Upload the zip to {BUNDLE}'


In [ ]:
# 3. Clone if new, then always fast-forward to the just-pushed main.

import os, subprocess, sys

REPO = Path('/content/thai-dsr')

if not (REPO / '.git').exists():

    subprocess.run(['git', 'clone', 'https://github.com/KankaveeRamsri/thai-dsr.git', str(REPO)], check=True)

os.chdir(REPO)

subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], check=True)

subprocess.run(['git', 'log', '-1', '--oneline'], check=True)

In [ ]:
# 4. Dependencies: preserve Colab's matching CUDA torch/torchaudio builds.
# Use subprocesses so package changes do not leave stale imports in the trainer.
import importlib.metadata as metadata
constraints = Path('/content/thai_dsr_constraints.txt')
constraints.write_text(''.join(f'{name}=={metadata.version(name)}\n' for name in ('torch', 'torchaudio')))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt',
                'PyYAML>=6.0', 'transformers==5.13.1', '-c', str(constraints)], check=True)
subprocess.run([sys.executable, '-c',
    'import torch, torchaudio, yaml; from transformers import Wav2Vec2Model; '
    'assert torch.cuda.is_available(); print(torch.__version__, torchaudio.__version__)'], check=True)


In [ ]:
# 5. Extract at repo root and verify every bundled file (SHA-256).
import hashlib, json, shutil, zipfile
local_zip = Path('/content/colab_transfer_bundle.zip')
shutil.copyfile(BUNDLE, local_zip)
with zipfile.ZipFile(local_zip) as archive:
    for entry in archive.infolist():
        target = (REPO / entry.filename).resolve()
        assert target.is_relative_to(REPO.resolve()), entry.filename
    archive.extractall(REPO)
local_zip.unlink()
manifest = json.loads((REPO / 'scripts/colab_bundle_manifest.json').read_text())
for entry in manifest['files']:
    p = REPO / entry['path']
    assert p.stat().st_size == entry['bytes'], p
    digest = hashlib.sha256()
    with p.open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024**2), b''):
            digest.update(block)
    assert digest.hexdigest() == entry['sha256'], f'Checksum mismatch: {p}'
print('Verified', len(manifest['files']), 'files; HiFi-GAN revision:', manifest['hifigan_revision'])
# The trainer confines outputs to this tree. A symlink sends writes directly to Drive.
output_link = REPO / 'results/checkpoints/joint_finetune'
if output_link.is_symlink():
    assert output_link.resolve() == CHECKPOINT_ROOT.resolve()
elif output_link.exists():
    raise RuntimeError(f'{output_link} already exists; preserve its contents before linking Drive.')
else:
    output_link.symlink_to(CHECKPOINT_ROOT, target_is_directory=True)
assert output_link.resolve() == CHECKPOINT_ROOT.resolve()
# Download the public frozen encoder automatically (about 1.2 GiB weights).
# The trainer reads this cache by model name; no local Mac cache is required.
os.environ['HF_HOME'] = '/content/thai_dsr_hf'
os.environ.pop('HF_HUB_OFFLINE', None)
subprocess.run([sys.executable, '-c',
    'from huggingface_hub import snapshot_download; '
    'snapshot_download("airesearch/wav2vec2-large-xlsr-53-th", '
    'allow_patterns=["config.json", "preprocessor_config.json", "model.safetensors", "pytorch_model.bin"])'], check=True)
subprocess.run([sys.executable, '-m', 'src.training.joint_finetune', '--help'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'tests.test_joint_finetune', '-v'], check=True)


In [ ]:
# 6. Fresh stability smoke (500 updates: 100 warmup + 200 ramp + 200 full GAN weight)

import time, uuid

from datetime import datetime

BASE = [sys.executable, '-u', '-m', 'src.training.joint_finetune',

        '--device', 'cuda', '--content-device', 'cuda',

        '--mapper-checkpoint', 'results/checkpoints/mapper_layer9.pt',

        '--manifest', 'data/manifest_w5.csv', '--splits', 'data/splits_w5.json',

        '--vocoder-init', 'thai', '--log-interval', '10']

def new_run(prefix):

    return output_link / (prefix + '_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8])

def run_training(extra, output):

    command = BASE + ['--output-dir', str(output)] + extra

    print(' '.join(command), flush=True)

    started = time.perf_counter()

    subprocess.run(command, cwd=REPO, check=True)

    elapsed = time.perf_counter() - started

    summary = json.loads((output / 'summary.json').read_text())

    print(f"Training steps/sec: {1 / summary['mean_seconds']:.4f}")

    print(f"End-to-end steps/sec (startup, validation, saving): {summary['steps'] / elapsed:.4f}")

    print('Validation trend:', json.dumps(summary['validation_trend'], indent=2))

    print(f"Estimated 2000-step compute time: {2000 * summary['mean_seconds'] / 3600:.2f} hours + validation/saving")

    return summary

smoke_output = new_run('stable_smoke500')

run_training(['--max-steps', '500', '--grad-clip', '1.0', '--gan-warmup-steps', '100',

              '--gan-ramp-steps', '200', '--content-weight', '0.5',

              '--loss-grad-interval', '100', '--checkpoint-interval', '250',

              '--validation-interval', '100', '--val-items', '3'], smoke_output)

## Full run / Resume

Smoke ต้องผ่าน finite-loss/gradient checks และการตรวจ validation trend ก่อนเริ่ม full run. Full 2,000 steps: clip=1, warmup=1000, ramp=1000, content=0.5; checkpoint ทุก 250 steps และ validation ทุก 1000 steps. บันทึกตรงใน Drive. `--max-steps` คือจำนวน step รวม. Resume เลือก checkpoint ล่าสุดและเขียนผลไปโฟลเดอร์ใหม่.

In [ ]:
# 7. FULL stable training; rerun to resume after an interruption.

MAX_STEPS = 2000

CHECKPOINT_INTERVAL = 250

RESUME_PATH = None  # Optional explicit path to a previous intact joint_XXXXXXXX.pt

candidates = list(CHECKPOINT_ROOT.glob('full_*/joint_*.pt'))

latest = Path(RESUME_PATH) if RESUME_PATH else max(

    candidates, key=lambda p: (int(p.stem.split('_')[-1]), p.stat().st_mtime), default=None)

step = 0

if latest is not None:

    state = torch.load(latest, map_location='cpu', weights_only=True, mmap=True)

    assert {'mapper', 'generator', 'mpd', 'msd', 'optim_g', 'optim_d', 'step',

            'sampling_rng', 'torch_rng'}.issubset(state), latest

    step = int(state['step'])

    del state

    print('Resume:', latest, 'step:', step)

if step >= MAX_STEPS:

    print('Target already reached. Increase MAX_STEPS to continue.')

else:

    output = new_run('full')

    extra = ['--max-steps', str(MAX_STEPS), '--checkpoint-interval', str(CHECKPOINT_INTERVAL),

             '--validation-interval', '1000', '--val-items', '3',

             '--grad-clip', '1.0', '--gan-warmup-steps', '1000', '--gan-ramp-steps', '1000',

             '--content-weight', '0.5', '--loss-grad-interval', '1000']

    if latest is not None:

        extra += ['--resume', str(latest)]

    print('Checkpoints saved directly to:', output.resolve())

    run_training(extra, output)